# 06 Next Best Action

**SYNTHETIC DEMO MODEL**. Reusable production logic lives in `ml/src`. This notebook inspects completed immutable evidence and never tunes against the test set.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'run_ml_pipeline.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from ml.src.utils.config import config, path
import pandas as pd
from IPython.display import display, Image
cfg = config()
reports = path(cfg, 'reports')

## Candidates, eligibility, business priority and reasons

Payroll, Travel, Digital Re-engagement, Savings cross-sell, Satisfaction Recovery, or No Action. FX is deferred. Contact exclusions and prior ownership override all scores. Priority is an index, not model probability or money.

In [ ]:
from ml.src.scoring.actions import recommend
manifest = json.loads((ROOT / 'data/reports/latest_ml.json').read_text())
folder = Path(manifest['gold_path'])
f = pd.read_parquet(folder / 'customer_ml_scores.parquet')
display(f.groupby('next_best_action').agg(customers=('customer_key', 'size'), mean_priority=('priority_score', 'mean')))
display(f[['next_best_action', 'next_best_action_score', 'reason_code', 'next_best_action_reason']].head(12))
display(pd.read_parquet(folder / 'candidate_action_scores.parquet').drop(columns='customer_key').head())

## Reproduce and limitations

Run the documented CLI to create a new experiment/version. Notebook execution only reads completed evidence. Models use synthetic behavior, weak holdout results do not justify real targeting, and effect measurement requires randomized treatment/control. See `docs/ml_leakage_prevention.md`, `docs/ml_responsible_ai.md` and the model cards.